# Aprendizado de Máquina — Aula prática 04

## Métodos Não Paramétricos (KNN)

**Gabriel Sanfins** &nbsp;·&nbsp; gabrielsanfins@id.uff.br

---

Nas Aulas 01 e 02 nós **escolhíamos a forma** de $r$ — uma reta, um polinômio de
grau 5 — e estimávamos alguns coeficientes. Von Neumann resumiu o desconforto
dessa postura: *"com quatro parâmetros eu ajusto um elefante; com cinco, faço a
tromba mexer."* A pergunta desta aula é a saída natural:

> **e se eu não quiser supor forma alguma para $r(x)$?**

A resposta é deixar os dados ditarem a forma, olhando só para a **vizinhança** do
ponto onde se quer prever. Vamos construir três estimadores assim — KNN,
Nadaraya–Watson e regressão local — e, o mais importante, medir o que eles cobram
por essa liberdade. O último experimento do notebook mostra que a conta pode ficar
salgada, e é a deixa para a Aula 05.

### Objetivos

Ao final deste notebook você deve ser capaz de:

- montar uma base de *splines* à mão e reconhecer o que a torna **local**;
- implementar KNN e ver o $k$ funcionar como botão de flexibilidade — só que ao
  contrário do grau do polinômio;
- escrever Nadaraya–Watson do zero e reencontrar o resultado do `statsmodels`;
- medir, em vez de aceitar, a afirmação *"o núcleo importa pouco, a janela
  importa muito"*;
- medir o **viés de fronteira** do grau 0 e a correção do grau 1 — e ver por que
  isso só aparece quando se mede viés, e não o erro de uma amostra;
- reproduzir o experimento do [ISLP] §3.5 em que a regressão linear, apesar de
  errada, ganha do KNN quando a dimensão cresce.

---
## 1. Importando os pacotes

In [ ]:
import numpy as np
import pandas as pd
from matplotlib.pyplot import subplots

Os objetos novos são o estimador KNN do `scikit-learn`, o `SplineTransformer`
(que monta uma base de *splines* pronta) e dois suavizadores do `statsmodels`,
que usaremos para conferir o que escrevermos à mão.

In [ ]:
import sklearn.linear_model as skl
import sklearn.model_selection as skm
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import SplineTransformer, StandardScaler

from statsmodels.nonparametric.kernel_regression import KernelReg
from statsmodels.nonparametric.smoothers_lowess import lowess

In [ ]:
import warnings
warnings.filterwarnings("ignore")

---
## 2. A mesma população de sempre

$$X \sim \mathrm{Unif}[-3,3], \qquad Y = \sin(1{,}5X) + 0{,}3X + \varepsilon,
  \qquad \varepsilon \sim N(0, 0{,}7^2).$$

Continuamos na população da Aula 01 pelo mesmo motivo de sempre: aqui conhecemos
$r$, então dá para separar viés de variância — que é exatamente o que esta aula
precisa medir.

In [ ]:
def r(x):
    return np.sin(1.5 * x) + 0.3 * x


SIGMA = 0.7
A, B = -3.0, 3.0
N_TR = 50


def amostra(n, rng):
    x = rng.uniform(A, B, size=n)
    return x, r(x) + rng.normal(0, SIGMA, size=n)


rng = np.random.default_rng(0)
x, y = amostra(N_TR, rng)
X = x.reshape(-1, 1)
grade = np.linspace(A, B, 500)

---
## 3. Bases: ganhando flexibilidade sem sair do linear

A primeira estratégia não precisa de método novo: basta **inventar atributos**.
Se ajustarmos

$$g(x) = \sum_{j=1}^{I} \beta_j\, \phi_j(x),$$

o modelo continua linear nos $\beta_j$ e sai por mínimos quadrados. Foi o que
fizemos na Aula 01 com $\phi_j(x)=x^j$.

Os *splines* trocam os monômios por uma base melhor. Um *spline* de grau $k$ com
nós $t_1 < \dots < t_p$ é um polinômio por partes de grau $k$, colado nos nós com
derivadas até ordem $k-1$ contínuas. A base mais fácil de entender é a **base
truncada**:

$$1,\ x,\ \dots,\ x^k,\qquad (x-t_j)_+^k,\ j=1,\dots,p,
\qquad \text{onde } (u)_+ = \max(u, 0).$$

O truque está na segunda metade. A função $(x-t_j)_+^k$ é **identicamente nula à
esquerda de $t_j$**: acrescentá-la não muda nada antes do nó, e permite mudar a
curvatura depois dele. E como ela e suas $k-1$ primeiras derivadas se anulam em
$t_j$, a emenda sai suave de graça. Cada nó compra exatamente um grau de
liberdade, colocado onde você quiser.

In [ ]:
def base_truncada(x, nos, grau=3):
    colunas = [x ** j for j in range(1, grau + 1)]
    colunas += [np.clip(x - t, 0, None) ** grau for t in nos]
    return np.column_stack(colunas)


nos = np.array([-1.5, 0.0, 1.5])
spline = skl.LinearRegression().fit(base_truncada(x, nos), y)

# um polinomio global com o MESMO numero de parametros, para comparar
I = base_truncada(x, nos).shape[1]
poli = skl.LinearRegression().fit(np.column_stack([x ** j for j in range(1, I + 1)]), y)

fig, ax = subplots(figsize=(5.6, 3.3))
ax.scatter(x, y, s=16, color="gray", alpha=0.8)
ax.plot(grade, r(grade), color="green", lw=1.8, label="r(x)")
ax.plot(grade, spline.predict(base_truncada(grade, nos)), color="crimson", lw=1.5,
        label=f"spline cubico, {len(nos)} nos")
ax.plot(grade, poli.predict(np.column_stack([grade ** j for j in range(1, I + 1)])),
        color="steelblue", lw=1.5, ls="--", label=f"polinomio global grau {I}")
for t in nos:
    ax.axvline(t, color="gray", lw=0.7, ls=":")
ax.set_xlabel("x"); ax.set_ylabel("y"); ax.set_ylim(-3, 3)
ax.legend(fontsize=7.5)
print(f"os dois modelos tem {I} parametros (fora o intercepto)")

Mesmo número de parâmetros, ajustes diferentes. A diferença que interessa não é
estética: é que o *spline* é **local** e o polinômio global não é. Dá para medir
isso. Vamos empurrar para cima o $y$ do ponto mais à direita e ver quanto cada
ajuste muda **do outro lado do domínio**.

In [ ]:
i_direita = np.argmax(x)
y_mexido = y.copy()
y_mexido[i_direita] += 3.0

esquerda = grade < -1.0

spline2 = skl.LinearRegression().fit(base_truncada(x, nos), y_mexido)
poli2 = skl.LinearRegression().fit(
    np.column_stack([x ** j for j in range(1, I + 1)]), y_mexido)

d_spline = np.abs(spline2.predict(base_truncada(grade, nos))
                  - spline.predict(base_truncada(grade, nos)))[esquerda]
G = np.column_stack([grade ** j for j in range(1, I + 1)])
d_poli = np.abs(poli2.predict(G) - poli.predict(G))[esquerda]

print(f"mexemos em y do ponto x = {x[i_direita]:.2f} (o mais a direita)")
print(f"mudanca maxima do ajuste em x < -1:")
print(f"   spline cubico  : {d_spline.max():.4f}")
print(f"   polinomio grau {I}: {d_poli.max():.4f}")

Um dado na ponta direita mexeu no ajuste do outro lado do domínio — mas mexeu
**bem menos** no *spline*. Não é zero porque a base truncada carrega os monômios
$x, x^2, x^3$, que são globais; a parte $(x-t_j)_+^3$, essa sim, é cega ao que
acontece antes do nó.

Na prática ninguém monta a base à mão: as bibliotecas usam **B-splines**, que
geram o mesmo espaço de funções mas com contas numericamente estáveis. *Mesmo
espaço* quer dizer ajuste idêntico, e isso é conferível.

In [ ]:
bs = SplineTransformer(degree=3, knots=np.r_[A, nos, B].reshape(-1, 1),
                       extrapolation="continue", include_bias=False)
modelo_bs = skl.LinearRegression().fit(bs.fit_transform(X), y)

f_trunc = spline.predict(base_truncada(grade, nos))
f_bs = modelo_bs.predict(bs.transform(grade.reshape(-1, 1)))
print(f"colunas: base truncada = {base_truncada(x, nos).shape[1]}, "
      f"B-spline = {bs.transform(X).shape[1]}")
print(f"maior diferenca entre os dois ajustes: {np.abs(f_trunc - f_bs).max():.3e}")

> **Sua vez.** Refaça o *spline* com 1 nó, depois com 10 nós igualmente espaçados
> em $[-3,3]$, e desenhe os três ajustes na mesma figura. Qual deles subajusta e
> qual persegue o ruído? O número de nós é mais um botão de complexidade — e a
> Aula 03 já ensinou como girá-lo.

---
## 4. $k$ vizinhos mais próximos

O estimador mais direto que existe: para prever em $x$, tire a média das
respostas dos $k$ pontos de treino mais próximos.

$$\widehat r(x) = \frac{1}{k}\sum_{i \in \mathcal{N}_x} Y_i .$$

Repare no que essa fórmula **não** tem: nenhum parâmetro a estimar, nenhuma
otimização. O "treino" do KNN consiste em guardar os dados.

In [ ]:
fig, axes = subplots(1, 3, figsize=(7.6, 2.7), sharey=True)
for ax, k, rotulo in zip(axes, [1, 9, 40], ["superajuste", "equilibrio", "subajuste"]):
    m = KNeighborsRegressor(n_neighbors=k).fit(X, y)
    ax.scatter(x, y, s=12, color="gray", alpha=0.7)
    ax.plot(grade, r(grade), color="green", lw=1.6, label="r(x)")
    ax.plot(grade, m.predict(grade.reshape(-1, 1)), color="crimson", lw=1.5,
            label="KNN")
    ax.set_title(f"k = {k} — {rotulo}", fontsize=9)
    ax.set_xlabel("x"); ax.set_ylim(-3.2, 3.2)
axes[0].set_ylabel("y"); axes[0].legend(fontsize=7.5, loc="upper left")

Compare com a Aula 01: é o mesmo trio subajuste–equilíbrio–superajuste, com o
botão girando **ao contrário**. Lá o grau *grande* era o flexível; aqui é o $k$
*pequeno*. Vale fixar isso, porque é fonte permanente de confusão.

E note o formato da curva com $k=1$: uma escada. O KNN é constante por partes por
construção — ele nunca vai produzir uma curva suave, por mais dados que receba.

### Preguiçoso de verdade

"O treino consiste em guardar os dados" não é força de expressão. Dá para
cronometrar.

In [ ]:
import time

rng_g = np.random.default_rng(1)
x_g, y_g = amostra(20_000, rng_g)
X_g = x_g.reshape(-1, 1)
X_novo = np.linspace(A, B, 20_000).reshape(-1, 1)

for nome, modelo in [("KNN (k=9)", KNeighborsRegressor(n_neighbors=9)),
                     ("regressao linear", skl.LinearRegression())]:
    t0 = time.perf_counter(); modelo.fit(X_g, y_g); t_fit = time.perf_counter() - t0
    t0 = time.perf_counter(); modelo.predict(X_novo); t_pred = time.perf_counter() - t0
    print(f"{nome:18s}  ajuste: {1000*t_fit:7.1f} ms   predicao: {1000*t_pred:7.1f} ms")

A regressão linear gasta o tempo dela no ajuste e depois prevê de graça: são
$d+1$ multiplicações. O KNN faz o contrário — e isso importa em produção, onde
normalmente se treina uma vez e se prevê milhões de vezes.

### Escolhendo $k$ com a ferramenta da Aula 03

In [ ]:
ks = np.array([1, 2, 3, 5, 8, 12, 20, 30, 40])
dobras = skm.KFold(5, shuffle=True, random_state=0)

cv = np.array([-skm.cross_val_score(KNeighborsRegressor(n_neighbors=k), X, y,
                                    cv=dobras, scoring="neg_mean_squared_error").mean()
               for k in ks])
k_cv = ks[np.argmin(cv)]

# o risco verdadeiro de cada k, que so a simulacao entrega
def risco_knn(ks, n_rep=200, semente=11):
    rng = np.random.default_rng(semente)
    x0 = np.linspace(A, B, 400)
    r0 = r(x0)
    X0 = x0.reshape(-1, 1)
    soma = np.zeros(len(ks))
    for _ in range(n_rep):
        xb, yb = amostra(N_TR, rng)
        for j, k in enumerate(ks):
            pred = KNeighborsRegressor(n_neighbors=k).fit(xb.reshape(-1, 1), yb).predict(X0)
            soma[j] += np.mean((pred - r0) ** 2)
    return soma / n_rep + SIGMA ** 2


verdade = risco_knn(ks)
print(f"k escolhido pela CV      : {k_cv}")
print(f"k que minimiza o risco   : {ks[np.argmin(verdade)]}")

In [ ]:
fig, ax = subplots(figsize=(5.4, 3.2))
ax.plot(ks, verdade, "o-", ms=4, color="crimson", label="risco verdadeiro")
ax.plot(ks, cv, "s-", ms=4, color="steelblue", label="validacao cruzada")
ax.axhline(SIGMA ** 2, ls="--", lw=1, color="green", label="sigma^2")
ax.set_xlabel("k (numero de vizinhos)"); ax.set_ylabel("erro quadratico medio")
ax.set_xticks(ks); ax.legend(fontsize=8)

> **Sua vez.** Refaça a curva de risco verdadeiro com `N_TR = 500` em vez de 50.
> O $k$ ótimo sobe ou desce? Pense antes de rodar: com mais dados, os 9 vizinhos
> mais próximos ficam **mais** próximos, e a média de mais respostas fica mais
> estável. Os dois efeitos puxam para o mesmo lado?

---
## 5. Nadaraya–Watson: trocando a janela dura por pesos suaves

O KNN dá peso $1/k$ aos $k$ vizinhos e $0$ a todo o resto. A transição é abrupta
— e é por isso que a curva vira escada. Nadaraya–Watson suaviza: usa **todas** as
observações, com pesos que decaem com a distância.

$$\widehat r(x) = \sum_{i=1}^n w_i(x)\,Y_i, \qquad
  w_i(x) = \frac{K(x, X_i)}{\sum_j K(x, X_j)} .$$

Cinco linhas de `numpy`, sem laço nenhum:

In [ ]:
def nucleo_gauss(u):
    return np.exp(-0.5 * u ** 2)


def nadaraya_watson(x_tr, y_tr, onde, h):
    W = nucleo_gauss((onde[:, None] - x_tr[None, :]) / h)
    return (W @ y_tr) / W.sum(axis=1)


print(f"em x = 0, o ajuste NW com h=0,35 vale "
      f"{nadaraya_watson(x, y, np.array([0.0]), 0.35)[0]:.4f}")
print(f"                        r(0) vale {r(0.0):.4f}")

Os quatro núcleos da tabela da aula, e o que de fato muda o ajuste:

In [ ]:
u = np.linspace(-2.2, 2.2, 800)
d = np.abs(u)
nucleos = {
    "uniforme": np.where(d <= 1, 1.0, 0.0),
    "gaussiano": np.exp(-0.5 * u ** 2),
    "triangular": np.where(d <= 1, 1 - d, 0.0),
    "Epanechnikov": np.where(d <= 1, 1 - u ** 2, 0.0),
}

fig, (ax1, ax2) = subplots(1, 2, figsize=(7.4, 2.9))
for nome, k in nucleos.items():
    ax1.plot(u, k / k.max(), label=nome)
ax1.set_xlabel("distancia ate x, em unidades de h")
ax1.set_ylabel("peso (reescalado)")
ax1.set_title("os quatro nucleos, todos com h=1", fontsize=9)
ax1.legend(fontsize=7.5)

ax2.scatter(x, y, s=10, color="gray", alpha=0.6)
ax2.plot(grade, r(grade), color="green", lw=1.6, label="r(x)")
for h, estilo in [(0.08, ":"), (0.35, "-"), (1.5, "--")]:
    ax2.plot(grade, nadaraya_watson(x, y, grade, h), color="crimson", ls=estilo,
             lw=1.4, label=f"h = {h}")
ax2.set_xlabel("x"); ax2.set_ylim(-3, 3)
ax2.set_title("mesmo nucleo, tres janelas", fontsize=9)
ax2.legend(fontsize=7, ncol=2, loc="upper left")

### "O núcleo importa pouco, a janela importa muito"

Essa frase aparece em todo texto sobre suavização, inclusive nas notas desta
aula. Vamos medi-la: para cada um dos quatro núcleos, escolhemos o melhor $h$ por
validação cruzada e comparamos o risco no ótimo.

In [ ]:
def nw_generico(x_tr, y_tr, onde, h, nome):
    u = (onde[:, None] - x_tr[None, :]) / h
    d = np.abs(u)
    if nome == "uniforme":
        W = (d <= 1).astype(float)
    elif nome == "gaussiano":
        W = np.exp(-0.5 * u ** 2)
    elif nome == "triangular":
        W = np.where(d <= 1, 1 - d, 0.0)
    else:
        W = np.where(d <= 1, 1 - u ** 2, 0.0)
    soma = W.sum(axis=1)
    soma[soma == 0] = np.nan          # nenhum vizinho dentro da janela
    return (W @ y_tr) / soma


hs = np.logspace(-1.3, 0.6, 30)
x0 = np.linspace(A, B, 400)
r0 = r(x0)

resultado = {}
for nome in nucleos:
    riscos = []
    for h in hs:
        rng_h = np.random.default_rng(12)
        soma = 0.0
        for _ in range(60):
            xb, yb = amostra(N_TR, rng_h)
            pred = nw_generico(xb, yb, x0, h, nome)
            soma += np.nanmean((pred - r0) ** 2)
        riscos.append(soma / 60 + SIGMA ** 2)
    riscos = np.array(riscos)
    resultado[nome] = (hs[np.argmin(riscos)], riscos.min(), riscos)

print(f"{'nucleo':14s} {'h otimo':>9s} {'risco no otimo':>16s}")
for nome, (h_ot, risco_ot, _) in resultado.items():
    print(f"{nome:14s} {h_ot:9.3f} {risco_ot:16.4f}")

piores = [v[2].max() for v in resultado.values()]
melhores = [v[1] for v in resultado.values()]
print(f"\nentre nucleos (no h otimo de cada): risco de {min(melhores):.4f} "
      f"a {max(melhores):.4f}  -> variacao de {100*(max(melhores)/min(melhores)-1):.1f}%")
print(f"dentro de um nucleo, variando h    : risco de {min(melhores):.4f} "
      f"a {max(piores):.4f}  -> variacao de {100*(max(piores)/min(melhores)-1):.0f}%")

In [ ]:
fig, ax = subplots(figsize=(5.4, 3.2))
for nome, (h_ot, _, riscos) in resultado.items():
    ax.plot(hs, riscos, lw=1.4, label=nome)
ax.axhline(SIGMA ** 2, ls="--", lw=1, color="green")
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("janela h"); ax.set_ylabel("risco")
ax.set_title("as quatro curvas quase se superpoem", fontsize=9)
ax.legend(fontsize=8)

> **A lição.** As quatro curvas são quase a mesma curva. Trocar de núcleo, com
> cada um no seu melhor $h$, muda o risco em **1,8%**; mexer em $h$ dentro de um
> mesmo núcleo muda em **126%** — setenta vezes mais. Faz sentido: os quatro
> núcleos são perfis parecidos de uma mesma ideia, dar mais peso a quem está
> perto, enquanto $h$ decide **o tamanho da vizinhança**, que é a quantidade que
> controla o balanço viés–variância.
>
> Repare também que o $h$ ótimo é diferente para cada núcleo (de $0{,}36$ a
> $0{,}88$): eles medem "largura" em unidades diferentes. Comparar dois núcleos
> com o *mesmo* $h$ compararia outra coisa.
>
> Consequência prática: gaste sua validação cruzada em $h$, não em comparar
> núcleos.

### Nadaraya–Watson é um mínimo quadrado local

Vale ver esta releitura, porque é ela que abre a próxima seção. O estimador NW em
um ponto $x$ é exatamente o $\widehat\beta_0$ que resolve

$$\widehat\beta_0 = \argmin_{\beta_0} \sum_{i=1}^n w_i(x)\,(Y_i - \beta_0)^2,$$

isto é, **uma constante ajustada por mínimos quadrados ponderados**, com os pesos
decaindo a partir de $x$. Confira:

In [ ]:
x_alvo = 0.7
w = nucleo_gauss((x - x_alvo) / 0.35)

# minimos quadrados ponderados de y contra uma coluna de 1s
beta0 = skl.LinearRegression(fit_intercept=False).fit(
    np.ones((N_TR, 1)), y, sample_weight=w).coef_[0]

print(f"constante local, por minimos quadrados ponderados: {beta0:.6f}")
print(f"formula de Nadaraya-Watson                       : "
      f"{nadaraya_watson(x, y, np.array([x_alvo]), 0.35)[0]:.6f}")

Se ajustar uma **constante** localmente já dá um método útil, nada impede ajustar
uma **reta**. É a mesma equação com mais uma coluna — e é a próxima seção.

---
## 6. Regressão local e o viés de fronteira

A regressão polinomial local troca a constante por um polinômio de grau $p$
ajustado em torno de cada $x$:

$$\widehat\beta_0,\dots,\widehat\beta_p = \argmin \sum_i w_i(x)
  \big(Y_i - \beta_0 - \beta_1 (x_i - x) - \dots\big)^2,$$

e a predição em $x$ é $\widehat\beta_0$. Com $p=0$ recuperamos Nadaraya–Watson.

Por que isso importa? Por causa das **bordas**. Tome $x$ na ponta direita do
suporte: a vizinhança dele só tem pontos de um lado, todos à esquerda. Se $r$
está subindo ali, a média local desses pontos é uma média de valores *menores*
que $r(x)$ — e a estimativa fica sistematicamente puxada para baixo. Não é ruído:
é **viés**, e ele não some por mais dados que você colete. A reta local escapa
porque ela estima a inclinação e extrapola com ela.

In [ ]:
def linear_local(x_tr, y_tr, onde, h):
    saida = np.empty(len(onde))
    for j, x0j in enumerate(onde):
        w = nucleo_gauss((x_tr - x0j) / h)
        Bm = np.c_[np.ones_like(x_tr), x_tr - x0j]     # centrada em x0j
        M = Bm.T @ (w[:, None] * Bm)
        saida[j] = np.linalg.solve(M, Bm.T @ (w * y_tr))[0]
    return saida

Agora o experimento. Duas escolhas dele merecem explicação.

A primeira: usamos o suporte $[-2,2]$ em vez de $[-3,3]$. O viés de fronteira é
proporcional a $r'(x)$ na borda, e $r'(\pm 3) \approx -0{,}02$ — quase plano.
Medir ali esconderia justamente o efeito. Já $r'(\pm 2) \approx -1{,}2$.

A segunda, mais importante: medimos **viés**, sobre 300 amostras, e não o erro de
uma amostra. A teoria diz que o grau 1 corrige o *viés* de fronteira; o erro de
uma realização mistura viés com ruído, e na borda o grau 1 é ruidoso. Vamos ver o
que acontece se você tentar o atalho.

In [ ]:
a, b = -2.0, 2.0
n_f, h_f, n_rep = 200, 0.35, 300
onde = np.linspace(a, b, 300)
r_onde = r(onde)

rng_f = np.random.default_rng(3)
est_nw = np.empty((n_rep, len(onde)))
est_ll = np.empty((n_rep, len(onde)))
for j in range(n_rep):
    xf = rng_f.uniform(a, b, size=n_f)
    yf = r(xf) + rng_f.normal(0, SIGMA, size=n_f)
    est_nw[j] = nadaraya_watson(xf, yf, onde, h_f)
    est_ll[j] = linear_local(xf, yf, onde, h_f)

media_nw, media_ll = est_nw.mean(axis=0), est_ll.mean(axis=0)
dp_nw, dp_ll = est_nw.std(axis=0), est_ll.std(axis=0)
vies_nw, vies_ll = np.abs(media_nw - r_onde), np.abs(media_ll - r_onde)
borda = (onde < a + 0.15 * (b - a)) | (onde > b - 0.15 * (b - a))

print(f"|vies| na fronteira : NW = {vies_nw[borda].mean():.4f}   "
      f"linear local = {vies_ll[borda].mean():.4f}")
print(f"|vies| no miolo     : NW = {vies_nw[~borda].mean():.4f}   "
      f"linear local = {vies_ll[~borda].mean():.4f}")
print(f"desvio na fronteira : NW = {dp_nw[borda].mean():.4f}   "
      f"linear local = {dp_ll[borda].mean():.4f}")

In [ ]:
fig, (ax1, ax2) = subplots(1, 2, figsize=(7.6, 3.0))
for ax in (ax1, ax2):
    ax.axvspan(a, a + 0.15 * (b - a), color="gray", alpha=0.12)
    ax.axvspan(b - 0.15 * (b - a), b, color="gray", alpha=0.12)

ax1.plot(onde, r_onde, color="green", lw=1.8, label="r(x)")
ax1.fill_between(onde, media_nw - dp_nw, media_nw + dp_nw, color="crimson", alpha=0.15)
ax1.plot(onde, media_nw, color="crimson", lw=1.5, label="NW (grau 0)")
ax1.fill_between(onde, media_ll - dp_ll, media_ll + dp_ll, color="steelblue", alpha=0.15)
ax1.plot(onde, media_ll, color="steelblue", lw=1.5, ls="--", label="linear local (grau 1)")
ax1.set_xlabel("x"); ax1.set_ylabel("estimativa media")
ax1.set_title(f"media de {n_rep} ajustes (+- 1 desvio)", fontsize=9)
ax1.legend(fontsize=7, loc="lower center")

ax2.plot(onde, vies_nw, color="crimson", lw=1.5, label="NW (grau 0)")
ax2.plot(onde, vies_ll, color="steelblue", lw=1.5, ls="--", label="linear local")
ax2.set_xlabel("x"); ax2.set_ylabel("|vies|")
ax2.set_title("o vies dispara so nas pontas", fontsize=9)
ax2.legend(fontsize=7.5)

No miolo as duas curvas de viés são indistinguíveis — a correção do grau 1 não é
uma melhora geral, é uma melhora **local, na borda**, e ali ela corta o viés a
menos da metade.

E ela não é de graça: o desvio-padrão na fronteira sobe junto. Estimar uma
inclinação a partir de poucos pontos de um lado só é tarefa ruidosa. Trocamos
viés por variância outra vez — a Aula 01, agora dentro de uma janela.

Agora o atalho que não funciona: comparar os dois pelo erro de **uma** amostra.

In [ ]:
rng_u = np.random.default_rng(3)
xu = rng_u.uniform(a, b, size=n_f)
yu = r(xu) + rng_u.normal(0, SIGMA, size=n_f)

erro_nw = np.abs(nadaraya_watson(xu, yu, onde, h_f) - r_onde)
erro_ll = np.abs(linear_local(xu, yu, onde, h_f) - r_onde)
print("erro absoluto medio na FRONTEIRA, em UMA amostra:")
print(f"   NW (grau 0)  : {erro_nw[borda].mean():.4f}")
print(f"   linear local : {erro_ll[borda].mean():.4f}")

> **A lição.** Numa amostra só, a comparação pode dar qualquer coisa — inclusive
> apontar o grau 0 como melhor. O que a teoria promete é sobre o **viés**, e viés
> é uma média sobre amostras: não existe dentro de um único conjunto de dados.
> Sempre que quiser verificar uma afirmação sobre viés, repita a simulação.

> **Sua vez.** Refaça a medição de viés com $h = 0{,}15$ e com $h = 0{,}8$. A
> vantagem do grau 1 na fronteira aumenta ou diminui quando a janela cresce?
> Relacione com o mecanismo: o viés de fronteira vem de a vizinhança ser
> assimétrica, e uma janela maior é mais assimétrica na borda.

---
## 7. As ferramentas prontas: `statsmodels`

Escrevemos os dois estimadores para saber o que eles fazem. Na prática usa-se o
`statsmodels`: `KernelReg` com `reg_type="lc"` é Nadaraya–Watson (*local
constant*) e com `"ll"` é a reta local; `lowess` é a versão robusta e clássica da
regressão local. Como sabemos o que as fórmulas fazem, dá para conferir.

In [ ]:
kr = KernelReg(endog=y, exog=x, var_type="c", reg_type="lc", bw=[0.35])
pred_kr, _ = kr.fit(grade)
pred_nosso = nadaraya_watson(x, y, grade, 0.35)
print(f"maior diferenca KernelReg x nossa funcao: "
      f"{np.abs(pred_kr - pred_nosso).max():.3e}")

Idênticos. Agora as três curvas juntas, com a mesma janela:

In [ ]:
lo = lowess(y, x, frac=0.3, return_sorted=True)

fig, ax = subplots(figsize=(5.6, 3.3))
ax.scatter(x, y, s=14, color="gray", alpha=0.7)
ax.plot(grade, r(grade), color="green", lw=1.8, label="r(x)")
ax.plot(grade, pred_nosso, color="crimson", lw=1.4, label="Nadaraya-Watson (h=0,35)")
ax.plot(grade, linear_local(x, y, grade, 0.35), color="steelblue", lw=1.4, ls="--",
        label="linear local (h=0,35)")
ax.plot(lo[:, 0], lo[:, 1], color="darkorange", lw=1.4, ls=":", label="lowess (frac=0,3)")
ax.set_xlabel("x"); ax.set_ylabel("y"); ax.set_ylim(-3, 3)
ax.legend(fontsize=7.5)

---
## 8. Quando o paramétrico, mesmo errado, ganha

Até aqui o não paramétrico só teve vantagens: não supõe forma, acompanha o que os
dados mostrarem. Falta a conta.

O experimento é o do [ISLP] §3.5 (a Figura 3.20 deles). A verdade é **não
linear** — é o nosso $r(x_1)$ — e depende de **uma única** covariável. As outras
$p-1$ são ruído puro, irrelevantes. A regressão linear está errada por
construção: ela nem consegue representar a senoide. Mesmo assim, vejamos quem
ganha à medida que $p$ cresce.

In [ ]:
def experimento_dimensao(p, ks, n_tr=300, n_te=4000, n_rep=30, semente=4):
    rng_d = np.random.default_rng(semente)
    erro_knn = np.zeros(len(ks))
    erro_lin = 0.0
    for _ in range(n_rep):
        Xtr = rng_d.uniform(A, B, size=(n_tr, p))
        ytr = r(Xtr[:, 0]) + rng_d.normal(0, SIGMA, size=n_tr)
        Xte = rng_d.uniform(A, B, size=(n_te, p))
        yte_medio = r(Xte[:, 0])           # comparamos com r, e somamos sigma^2

        lin = skl.LinearRegression().fit(Xtr, ytr)
        erro_lin += np.mean((lin.predict(Xte) - yte_medio) ** 2)
        for j, k in enumerate(ks):
            knn = KNeighborsRegressor(n_neighbors=k).fit(Xtr, ytr)
            erro_knn[j] += np.mean((knn.predict(Xte) - yte_medio) ** 2)
    return erro_knn / n_rep + SIGMA ** 2, erro_lin / n_rep + SIGMA ** 2


ks_d = np.array([1, 2, 3, 5, 10, 20, 50, 100, 200])
ps = [1, 2, 4, 10, 20]
saida = {p: experimento_dimensao(p, ks_d) for p in ps}

# a referencia: o preditor constante, que nao aprende nada
xx = np.linspace(A, B, 200_000)
risco_constante = SIGMA ** 2 + r(xx).var()

linha = []
for p in ps:
    knn_p, lin_p = saida[p]
    linha.append({"p": p, "melhor KNN": knn_p.min(), "k otimo": ks_d[np.argmin(knn_p)],
                  "regressao linear": lin_p,
                  "vantagem do KNN": lin_p / knn_p.min()})
print(f"risco de quem chuta a media (nao aprende nada): {risco_constante:.4f}\n")
pd.DataFrame(linha).set_index("p").round(3)

In [ ]:
fig, axes = subplots(1, len(ps), figsize=(11, 2.6), sharey=True)
for ax, p in zip(axes, ps):
    knn_p, lin_p = saida[p]
    ax.plot(1 / ks_d, knn_p, "o-", ms=3.5, color="green", label="KNN")
    ax.axhline(lin_p, ls="--", color="black", lw=1.2, label="regressao linear")
    ax.axhline(risco_constante, ls=":", color="gray", lw=1.2, label="chutar a media")
    ax.set_xscale("log"); ax.set_title(f"p = {p}", fontsize=9)
    ax.set_xlabel("1/k")
axes[0].set_ylabel("risco"); axes[0].legend(fontsize=7)

Com $p=1$ o KNN ganha com folga — a verdade é curva e ele acompanha, cortando
quase pela metade o risco da reta. A vantagem então **mingua monotonicamente**:
1,9× em $p=1$, 1,3× em $p=4$, e a partir de $p=10$ ela vira desvantagem. Em
$p=20$ a regressão linear ganha, apesar de não conseguir **nem representar** a
função verdadeira.

O motivo não é a regressão linear ter melhorado. Ela nem sabe que as variáveis 2
a $p$ são lixo, e paga um pouco de variância por cada uma. O que acontece é que o
KNN piora muito mais rápido. Com 300 pontos espalhados em $[-3,3]^{20}$, os
"$k$ vizinhos mais próximos" de um ponto não são próximos de coisa nenhuma: a
média local deixa de ser local, e o viés explode.

E olhe a linha pontilhada: em $p=20$ os dois métodos já estão a menos de 15% do
risco de quem simplesmente chuta a média, sem olhar covariável nenhuma. Com essa
dimensão e essa amostra, quase não há aprendizado a extrair — e a curva do KNN
fica achatada, sinal de que nenhum $k$ resolve o problema.

Isso tem nome — **maldição da dimensionalidade** — e é a Aula 05 inteira. O que
esta figura antecipa é a moral: *supor uma forma errada pode custar menos que não
supor forma alguma*, e quanto maior $d$, mais isso vale.

---
## 9. No mundo real: KNN no `superconductivity.csv`

Duas advertências práticas para fechar. A primeira é a que a Aula 05 vai
formalizar: aqui $d = 81$, e você já sabe o que isso significa para um método de
vizinhança. A segunda é a escala.

In [ ]:
import os

_nome = "superconductivity.csv"
_local = os.path.join("..", "..", "recursos", "dados", _nome)   # repositorio clonado
_url = ("https://raw.githubusercontent.com/HugoCarvalhoUFRJ/ap-maq/"
        "refs/heads/refactoring-baby/recursos/dados/") + _nome  # fallback (ex.: Colab)
_fonte = _local if os.path.exists(_local) else _url

df = pd.read_csv(_fonte)
Xs = df.drop(columns="critical_temp").values
ys = df["critical_temp"].values

# o KNN gasta o tempo na PREDICAO, e ela custa O(n_treino) por consulta.
# Uma subamostra de 4000 pontos mantem o notebook rodando em segundos.
rng_s = np.random.default_rng(0)
sub = rng_s.choice(len(ys), size=4000, replace=False)
X_tr, X_te, y_tr, y_te = skm.train_test_split(Xs[sub], ys[sub], test_size=0.3,
                                              random_state=0)
print("treino:", X_tr.shape, "  teste:", X_te.shape)

### A escala não é preciosismo

As 81 colunas estão em unidades completamente diferentes — massa atômica em
unidades de massa atômica, condutividade térmica em W/(m·K), e por aí. A
distância euclidiana soma tudo, então a coluna de maior amplitude decide sozinha
quem é vizinho de quem.

In [ ]:
amplitudes = Xs.max(axis=0) - Xs.min(axis=0)
print(f"amplitude das 81 colunas: de {amplitudes.min():.3g} a {amplitudes.max():.3g}")
print(f"razao entre a maior e a menor: {amplitudes.max() / amplitudes.min():.3g}")

from sklearn.metrics import mean_squared_error

sem_escala = KNeighborsRegressor(n_neighbors=9).fit(X_tr, y_tr)
com_escala = Pipeline([("escala", StandardScaler()),
                       ("knn", KNeighborsRegressor(n_neighbors=9))]).fit(X_tr, y_tr)

print(f"\nEQM no teste, KNN k=9 SEM padronizar: "
      f"{mean_squared_error(y_te, sem_escala.predict(X_te)):.2f}")
print(f"EQM no teste, KNN k=9 COM padronizar: "
      f"{mean_squared_error(y_te, com_escala.predict(X_te)):.2f}")

E a padronização precisa acontecer **dentro** do `Pipeline`, não antes da
validação cruzada: assim ela é refeita em cada dobra, usando só o treino daquela
dobra. É a regra da Aula 03 contra vazamento, e a Aula 07 volta ao assunto.

In [ ]:
modelo = Pipeline([("escala", StandardScaler()), ("knn", KNeighborsRegressor())])
busca = skm.GridSearchCV(modelo, {"knn__n_neighbors": [1, 3, 5, 9, 15, 25, 50]},
                         cv=skm.KFold(5, shuffle=True, random_state=0),
                         scoring="neg_mean_squared_error")
busca.fit(X_tr, y_tr)

# a Ridge da Aula 03, tambem com o hiperparametro escolhido por CV -- comparar
# um modelo ajustado com um modelo no chute nao diria nada
ridge = skm.GridSearchCV(
    Pipeline([("escala", StandardScaler()), ("ridge", skl.Ridge())]),
    {"ridge__alpha": np.logspace(-2, 4, 20)},
    cv=skm.KFold(5, shuffle=True, random_state=0),
    scoring="neg_mean_squared_error").fit(X_tr, y_tr)

print(f"melhor k    : {busca.best_params_['knn__n_neighbors']}")
print(f"melhor alpha: {ridge.best_params_['ridge__alpha']:.4g}")
print(f"\nEQM no teste, KNN   : {mean_squared_error(y_te, busca.predict(X_te)):.2f}")
print(f"EQM no teste, Ridge : {mean_squared_error(y_te, ridge.predict(X_te)):.2f}")
print(f"variancia de y      : {y_te.var():.2f}")

Aqui o KNN ganha, e ganha de longe — em $d = 81$, logo depois de a Seção 8 ter
mostrado o KNN perdendo em $d = 20$. Isso não é contradição, é a informação mais
útil desta seção.

A conta da Seção 8 usava 20 covariáveis **independentes**: cada uma acrescentava
uma direção nova ao espaço, e o volume a preencher multiplicava. As 81 colunas do
`superconductivity.csv` são outra coisa. São todas funções das mesmas
propriedades atômicas dos elementos da fórmula química — média, média ponderada,
média geométrica, entropia, faixa, desvio-padrão de cada propriedade. Elas são
fortemente redundantes, e os materiais não ocupam $[\,\cdot\,]^{81}$: ficam numa
região de dimensão efetiva muito menor.

O que mata o KNN não é o número de colunas, é a **dimensão efetiva** dos dados. A
Aula 05 dá nome a essa distinção e mostra as duas rotas de fuga: quando poucas
covariáveis importam (esparsidade) e quando muitas covariáveis descrevem poucas
direções (redundância).

> **Sua vez.** Repita a comparação usando só as 5 colunas mais correlacionadas com
> `critical_temp`, em vez das 81. O KNN melhora ou piora? E a Ridge? Depois de
> responder, olhe de novo a figura da Seção 8 — você acabou de andar para a
> esquerda naquele painel.

---
## Resumo

| Conceito | Onde apareceu | O que vimos |
|---|---|---|
| base truncada | §3 | $(x-t_j)_+^k$ é nula antes do nó — daí a localidade; um dado na ponta mexe pouco no outro lado |
| B-splines | §3 | mesmo espaço da base truncada: ajuste idêntico a $10^{-14}$ |
| KNN | §4 | $k$ **pequeno** é o flexível; a curva é escada por construção |
| custo | §4 | preguiçoso: ajusta em milissegundos, prevê devagar |
| Nadaraya–Watson | §5 | média ponderada; escrito em 5 linhas, bate com `KernelReg` a $10^{-15}$ |
| núcleo × janela | §5 | trocar de núcleo muda o risco em 1,8%; mudar $h$, em 126% |
| NW é local | §5 | é a constante dos mínimos quadrados ponderados |
| viés de fronteira | §6 | grau 1 corta o viés na borda a menos da metade, e paga em variância |
| viés precisa de repetições | §6 | numa amostra só, a comparação chega a inverter |
| dimensão | §8 | a vantagem do KNN cai de 1,9× ($p=1$) a menos de 1 ($p\ge10$) |
| escala | §9 | sem `StandardScaler`, a coluna de maior amplitude decide quem é vizinho |
| dimensão efetiva | §9 | em $d=81$ **redundantes** o KNN volta a ganhar — o que conta não é o número de colunas |

**Leitura recomendada.** [AME] §4.1–4.2 (séries e *splines*, com a base truncada
que montamos na Seção 3), §4.3 (KNN e a Figura 4.2 sobre o efeito de $k$), §4.4
(Nadaraya–Watson, com a tabela de núcleos e a Figura 4.4, que é a versão deles do
que medimos na Seção 5) e §4.5 (regressão polinomial local). [ISLP] §3.5 — o
experimento da Seção 8 é a Figura 3.20 deles — e o Capítulo 7 inteiro, em especial
§7.4–7.6.

**Para praticar.** `Lista teorica 04.pdf` (teórica, com gabarito) e
`Lista prática 04.ipynb` (prática, para completar as lacunas), nesta mesma
pasta.

**A seguir.** A Aula 05 explica, com teoria, o que a Seção 8 mostrou com uma
simulação: por que a taxa de convergência de todo método de vizinhança degrada
como $n^{-2/(2+d)}$, e o que ainda dá para fazer quando $d$ é grande.